# jfinance quickstart

[jfinance](https://github.com/sgawa/jfinance) reads corporate disclosure data filed with
**EDINET**, the Financial Services Agency of Japan's electronic disclosure system.

No registration and no API key. The API follows yfinance, so most yfinance code runs unchanged.

Coverage: annual, semi-annual and quarterly securities reports and large shareholding
reports **from fiscal 2016 onwards**.


## Install

In [ ]:
!pip install -q jfinance


In [ ]:
import jfinance as jf
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 12)

jf.__version__


## A company

A ticker is a securities code (`7203.T`), an EDINET code (`E02144`), an ISIN
(`JP3633400001`) or an EDINET fund code (`G02925`).

In [ ]:
t = jf.Ticker("7203.T")     # Toyota

{k: t.info[k] for k in
 ("shortName", "sector", "industry", "fullTimeEmployees",
  "fiscalYearEnd", "sharesOutstanding", "returnOnEquity")}


## Financial statements

`financials`, `balance_sheet` and `cashflow` return line items as rows and period end
dates as columns, newest first. Amounts are in yen.

Unlike yfinance, **every fiscal year is returned by default**, not just the last four.

In [ ]:
t.financials.iloc[:8, :4]


In [ ]:
print(t.financials.shape, t.balance_sheet.shape, t.cashflow.shape)

# Quarterly reporting was abolished in April 2024 and replaced by semi-annual
# reporting, so quarterly figures stop at fiscal 2023.
print(t.quarterly_financials.shape, t.get_income_stmt(freq="semiannual").shape)


### Items Yahoo Finance has no name for

Ordinary income, book value per share, headcount, and the line items particular to
banks and insurers.

In [ ]:
t.get_jp_financials().iloc[:8, :4]


### Every line, as filed

`get_statements()` returns the statements in the order and hierarchy of the filing
itself, taken from the XBRL presentation linkbase.

In [ ]:
t.get_statements()[["Role", "Label", "Local Name", "Depth"]].head(10)


## Shareholders

Japanese disclosure splits this across several filings, and jfinance exposes each one
separately rather than merging them.

In [ ]:
t.major_shareholders[["Holder", "pctHeld", "Shares"]].head()


In [ ]:
t.major_holders      # ownership breakdown by holder type


In [ ]:
# Filers of large shareholding reports (the 5% rule), including companies and individuals
jf.Ticker("8035.T").large_holders[["Holder", "pctHeld", "Date Filed"]].head()


## Officers and pay

Officers are disclosed once a year, in the annual report. Individual remuneration is
disclosed only where it reaches 100 million yen.

In [ ]:
t.officers[["Name", "Title", "Birth Date", "Shares Held"]].head()


In [ ]:
t.officer_compensation.head()


## Segments, workforce, filings

In [ ]:
t.segments.query("Metric == 'seg_revenue'")[
    ["Fiscal Year", "Segment Label", "Value"]].head(8)


In [ ]:
t.employees.iloc[:5, :8]


In [ ]:
t.get_filings(types=["120"], limit=5)[["Filing Date", "Title", "Fiscal Year"]]


### Greenhouse gas emissions

Tagged in XBRL from the fiscal 2024 annual reports onwards, so many companies are
still empty.

In [ ]:
jf.Ticker("9432.T").emissions     # NTT


## Finding companies

In [ ]:
pd.DataFrame(jf.Search("toyota").quotes)[["symbol", "shortname", "quoteType"]]


In [ ]:
jf.JpSector.all().head()          # the 17 TOPIX sectors


In [ ]:
jf.JpSector("automobiles-transportation-equipment").top_companies.head()


### Screener

Filters on disclosed figures. Written the same way as yfinance's `screen`.

Fields derived from share prices — market capitalisation, P/E, share price — are not
available, because EDINET does not carry them.

In [ ]:
res = jf.edinet_screen(
    jf.EdinetQuery("gt", ["roe", 0.2]),
    sortField="revenue",
    size=10,
)
print(res["total"], "companies matched")
pd.DataFrame(res["quotes"])[["symbol", "shortName", "roe", "revenue"]]


In [ ]:
# What can be queried
q = jf.EdinetQuery("gt", ["roe", 0.2])
{k: len(v) for k, v in q.valid_fields.items()}


## Investment trusts

25 items per reporting date. Identify a fund by its EDINET fund code (starting with
`G`), or by securities code if it is exchange traded.

In [ ]:
jf.Ticker("1306.T").get_fund_financials().iloc[:8, :4]


## Filings on a given day

Across all companies.

In [ ]:
jf.FilingCalendar("2026-06-25").get_filings(types=["120"], limit=5)[
    ["Filing Date", "Title"]]


## Japanese

Company and industry names come back in English by default.

Business descriptions and the names of officers and shareholders are returned in
Japanese under either setting, because EDINET holds no English original.

In [ ]:
jf.config.locale.lang = "ja-JP"

t = jf.Ticker("7203.T")
print(t.info["shortName"], "/", t.info["sector"])


## What is not here

Share prices, dividend history, splits, options, analyst estimates, news, earnings
calendars and ESG scores do not exist in EDINET. Those yfinance attributes are **not
defined**, so accessing one raises `AttributeError`.

In [ ]:
for name in ("history", "dividends", "splits", "news", "recommendations"):
    print(f"{name:16} {'present' if hasattr(t, name) else 'not defined'}")


## Please read

**Amendments to filings past their EDINET public inspection period cannot be
retrieved, and may therefore not be reflected.** Older fiscal years are more likely to
retain pre-amendment values.

In [ ]:
print(jf.NOTICE_CORRECTIONS)


---

Data source and terms:

```
出典：EDINET閲覧（提出）サイト（https://disclosure2.edinet-fsa.go.jp/）、
      PDL1.0（https://www.digital.go.jp/resources/open_data/public_data_license_v1.0）
EDINET閲覧（提出）サイト（https://disclosure2.edinet-fsa.go.jp/）をもとに jfinance 作成
```

The same notice is in the `X-JF-Notice` header of every response.
Documentation: <https://jfnc.org/>
